In [1]:
from general.FlopsCalculator import FlopsCalculator
from spconv.pytorch import SparseConvTensor
from spconv.pytorch import SparseConv3d, SubMConv3d
import torch

In [2]:
def generate_distant_sparse_tensor():
    features = torch.ones(
        (4, 1),
        device='cuda',
        dtype=torch.float32
    )

    indices = torch.tensor([
        [0, 25, 25, 25],
        [0, 75, 75, 75],
        [0, 50, 50, 50],
        [0, 25, 25, 75],
    ], device='cuda', dtype=torch.int32)

    x = SparseConvTensor(
        features,
        indices,
        [100, 100, 100],
        1
    )

    return x

def generate_near_sparse_tensor():
    features = torch.ones(
        (4, 1),
        device='cuda',
        dtype=torch.float32
    )

    indices = torch.tensor([
        [0, 50, 50, 50],
        [0, 50, 51, 50],
        [0, 51, 50, 50],
        [0, 51, 51, 50],
    ], device='cuda', dtype=torch.int32)

    x = SparseConvTensor(
        features,
        indices,
        [100, 100, 100],
        1
    )

    return x

In [3]:
flops_calculator = FlopsCalculator({
    'test': {}
})

In [4]:
conv = SparseConv3d(1, 1, kernel_size=3, bias=False, indice_key='test_conv3d_distant').cuda()

conv.register_forward_hook(
    flops_calculator.conv3d_hook(
        'test',
        'test_conv3d_distant'
    )
)

y = conv(generate_distant_sparse_tensor())

In [5]:
submconv = SubMConv3d(1, 1, kernel_size=3, bias=False, indice_key='test_submconv3d_distant').cuda()

submconv.register_forward_hook(
    flops_calculator.conv3d_hook(
        'test',
        'test_submconv3d_distant'
    )
)

y = submconv(generate_distant_sparse_tensor())

In [6]:
conv = SparseConv3d(1, 1, kernel_size=3, bias=False, indice_key='test_conv3d_near').cuda()

conv.register_forward_hook(
    flops_calculator.conv3d_hook(
        'test',
        'test_conv3d_near'
    )
)

y = conv(generate_near_sparse_tensor())

In [7]:
submconv = SubMConv3d(1, 1, kernel_size=3, bias=False, indice_key='test_submconv3d_near').cuda()

submconv.register_forward_hook(
    flops_calculator.conv3d_hook(
        'test',
        'test_submconv3d_near'
    )
)

y = submconv(generate_near_sparse_tensor())

In [8]:
flops_calculator.per_layer_stats

{'test': {'test_conv3d_distant': {'name': 'test_conv3d_distant',
   'channels_in': 1,
   'channels_out': 1,
   'kernel_volume': 27,
   'voxels_active': 108,
   'MACs': 108,
   'FLOPs': 216.0,
   'count': 1,
   'MACs_dense': 2916,
   'FLOPs_dense': 5832.0},
  'test_submconv3d_distant': {'name': 'test_submconv3d_distant',
   'channels_in': 1,
   'channels_out': 1,
   'kernel_volume': 27,
   'voxels_active': 4,
   'MACs': 4,
   'FLOPs': 8.0,
   'count': 1,
   'MACs_dense': 108,
   'FLOPs_dense': 216.0},
  'test_conv3d_near': {'name': 'test_conv3d_near',
   'channels_in': 1,
   'channels_out': 1,
   'kernel_volume': 27,
   'voxels_active': 48,
   'MACs': 108,
   'FLOPs': 216.0,
   'count': 1,
   'MACs_dense': 1296,
   'FLOPs_dense': 2592.0},
  'test_submconv3d_near': {'name': 'test_submconv3d_near',
   'channels_in': 1,
   'channels_out': 1,
   'kernel_volume': 27,
   'voxels_active': 4,
   'MACs': 16,
   'FLOPs': 32.0,
   'count': 1,
   'MACs_dense': 108,
   'FLOPs_dense': 216.0}}}